In [1]:
import pandas as pd
import requests as requests

In [2]:
# Getting JSON data
# Extracted player data JSON link from the official NWSL website by inspect element -> Network -> Fetch/XHR

url = "https://api-sdp.nwslsoccer.com/v1/nwsl/football/seasons/nwsl::Football_Season::fad050beee834db88fa9f2eb28ce5a5c/stats/players?locale=en-US&category=general&role=all&direction=desc&page=1&pageNumElement=400"

session = requests.Session()
response = session.get(url)

data = response.json()

In [3]:
#Getting player stats data from JSON file

df = pd.json_normalize(data["players"])
df = df.explode("stats")
print(df.columns)

Index(['stats', 'rankLabel', 'playerId', 'providerId', 'bibNumber',
       'roleLabel', 'role', 'mediaFirstName', 'mediaLastName', 'shirtName',
       'shortName', 'displayName', 'nationality', 'nationalityIsoCode',
       'apiCallRequestTime', 'team.teamId', 'team.providerId',
       'team.shortName', 'team.officialName', 'team.acronymName',
       'team.acronymNameLocalized', 'team.isTeamFake', 'team.mediaName',
       'team.mediaShortName', 'team.countryCode', 'team.teamType',
       'team.overallSummary', 'team.stadium', 'team.allSeasonImagery',
       'team.editorial.social.facebook', 'team.editorial.social.instagram',
       'team.editorial.social.x', 'team.editorial.social.tikTok',
       'team.editorial.social.youTube', 'team.editorial.social.linkedIn',
       'team.editorial.websiteUrl', 'team.editorial.shopUrl',
       'team.editorial.ticketsUrl', 'team.editorial.clubPrimaryColour',
       'team.editorial.clubSecondaryColour', 'team.editorial.clubTextColour',
       'editoria

In [4]:
# Cleaning Data
# Turning Unstructured to Structured data

stats = pd.json_normalize(df["stats"])
df = pd.concat([df.drop(columns=["stats"]), stats], axis=1)
print(df.columns)

Index(['rankLabel', 'playerId', 'providerId', 'bibNumber', 'roleLabel', 'role',
       'mediaFirstName', 'mediaLastName', 'shirtName', 'shortName',
       'displayName', 'nationality', 'nationalityIsoCode',
       'apiCallRequestTime', 'team.teamId', 'team.providerId',
       'team.shortName', 'team.officialName', 'team.acronymName',
       'team.acronymNameLocalized', 'team.isTeamFake', 'team.mediaName',
       'team.mediaShortName', 'team.countryCode', 'team.teamType',
       'team.overallSummary', 'team.stadium', 'team.allSeasonImagery',
       'team.editorial.social.facebook', 'team.editorial.social.instagram',
       'team.editorial.social.x', 'team.editorial.social.tikTok',
       'team.editorial.social.youTube', 'team.editorial.social.linkedIn',
       'team.editorial.websiteUrl', 'team.editorial.shopUrl',
       'team.editorial.ticketsUrl', 'team.editorial.clubPrimaryColour',
       'team.editorial.clubSecondaryColour', 'team.editorial.clubTextColour',
       'editorial.playerR

In [5]:
print(df.head(5))

  rankLabel                                           playerId  \
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   

                              providerId bibNumber   roleLabel  role  \
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   

  mediaFirstName mediaLastName shirtName shortName  ...  \
0          Julie         Doyle            J. Doyle  ...   
0          Julie         Doyle            J. Doyle  

In [6]:
# Pivoting statsId column values

df_pivot = df.pivot_table(
    index="playerId",
    columns="statsLabel",
    values="statsValue"
).reset_index()
print(df_pivot.columns)

Index(['playerId', 'Accurate pass percentage', 'Aerial Duels',
       'Aerial Duels Won Percentage', 'Aerial Duels lost', 'Aerial Duels won',
       'Aerials Won Percentage', 'Appearances', 'Assists',
       'Assists (Intentional)',
       ...
       'Unsuccessful Passes Own Half', 'Unsuccessful Short Passes',
       'Unsuccessful lay-offs', 'Winning Goal', 'XGEfficiency', 'Xg',
       'Yellow Cards', 'Yellow Red Cards', 'Yellow cards', 'corners'],
      dtype='str', name='statsLabel', length=174)


In [7]:
print(df_pivot.head(5))

statsLabel                                           playerId  \
0           nwsl::Football_Player::0021a2896ee54ef191fb324...   
1           nwsl::Football_Player::01429efe66554be8bc7f86e...   
2           nwsl::Football_Player::021370e22b6846a48f5e988...   
3           nwsl::Football_Player::0441c191624b4049911e4bb...   
4           nwsl::Football_Player::04a1aad603eb46dd9c349c2...   

statsLabel  Accurate pass percentage  Aerial Duels  \
0                               61.0          31.0   
1                               75.0          27.0   
2                               84.0          27.0   
3                               69.0           6.0   
4                               68.0          36.0   

statsLabel  Aerial Duels Won Percentage  Aerial Duels lost  Aerial Duels won  \
0                                 54.84               14.0              17.0   
1                                 59.26               11.0              16.0   
2                                 77.78     

In [8]:
# Cleaned df
df_final = pd.merge(df.drop(columns=['statsId', 'statsLabel',
       'statsLabelAbbreviation', 'statsValue', 'statsUnit',
       'statsUnitAbbreviation']), df_pivot, on="playerId")
df_final = df_final.drop_duplicates(["playerId"]).reset_index(drop=True)
print(df_final.columns)

Index(['rankLabel', 'playerId', 'providerId', 'bibNumber', 'roleLabel', 'role',
       'mediaFirstName', 'mediaLastName', 'shirtName', 'shortName',
       ...
       'Unsuccessful Passes Own Half', 'Unsuccessful Short Passes',
       'Unsuccessful lay-offs', 'Winning Goal', 'XGEfficiency', 'Xg',
       'Yellow Cards', 'Yellow Red Cards', 'Yellow cards', 'corners'],
      dtype='str', length=214)


In [9]:
print(df_final.head(5))

  rankLabel                                           playerId  \
0      None  nwsl::Football_Player::4cb80ed654ff46e89167791...   
1      None  nwsl::Football_Player::a86e9ba2f4c44ce789c593b...   
2      None  nwsl::Football_Player::ff16e70c73b943e38662bc6...   
3      None  nwsl::Football_Player::0021a2896ee54ef191fb324...   
4      None  nwsl::Football_Player::021370e22b6846a48f5e988...   

                              providerId bibNumber   roleLabel  role  \
0  opta:Player:3sgg664ay83kxhe17ogbke978        20  Midfielder     3   
1  opta:Player:d9dirjpaqkq3gga2zyeykcv6d         1  Goalkeeper     1   
2  opta:Player:9yvkpwm78jaqr321imfc3rrbo         6    Defender     2   
3  opta:Player:e2obgph6szyo04em3qr94g1qt        22     Forward     4   
4  opta:Player:bilgoekg6gqwr3qbp6dp0j1zp         4    Defender     2   

   mediaFirstName   mediaLastName     shirtName     shortName  ...  \
0           Julie           Doyle                    J. Doyle  ...   
1          Aubrey       Kingsb

In [10]:
# QC

In [13]:
print(df_final.shape) # Matching no of rows in json data

(400, 214)


In [14]:
df.loc[df["playerId"].isnull()==True] # No nulls in pk

,rankLabel,playerId,providerId,bibNumber,roleLabel,role,mediaFirstName,mediaLastName,shirtName,shortName,...,team.editorial.clubPrimaryColour,team.editorial.clubSecondaryColour,team.editorial.clubTextColour,editorial.playerRoleWithinTeam,statsId,statsLabel,statsLabelAbbreviation,statsValue,statsUnit,statsUnitAbbreviation
